# Predict External Kidney Slides and Save PDF Figures

Run this notebook from top to bottom.

It keeps only the core workflow:
1. predict the selected external kidney slides with one trained model
2. keep the existing per-slide 4-gene selection logic
3. take the union of the two 4-gene lists
4. save one spatial summary PDF per slide on the tissue background

Saved outputs go to `results/other_kidney/<slide_id>/<model_key>/figures/`.
The final cell also writes a union-gene JSON file and a figure-summary CSV file.


In [1]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

ROOT = Path(r"E:\Morpho-FM")
sys.path.append(str(ROOT / "src"))

from st_pipeline.constants import KEYS
from st_pipeline.data.collate import mil_collate
from st_pipeline.data.gene_vocab import load_gene_vocab, map_genes_to_vocab
from st_pipeline.model.morpho_cellfm_mil import MorphoCellfmMIL
from st_pipeline.superres.feature_cache import normalize_slide

try:
    from IPython.display import display
except ImportError:
    display = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None, leave=True):
        if iterable is None:
            return range(total or 0)
        return iterable

CACHE_ROOT = ROOT / "data" / "other_kidney_cache"
RESULT_ROOT = ROOT / "results" / "other_kidney"
GENE_VOCAB = ROOT / "assets" / "cellfm" / "expand_gene_info.csv"
CELLFM_CKPT = ROOT / "checkpoints" / "CellFM" / "CellFM_80M_weight.pt"

SLIDE_IDS = ["GSM6415706_GLMF1", "GSM6415706_GLMF2"]
MODEL_SPECS = {
    "multi": ROOT / "checkpoints" / "kidney_hest" / "st_superres" / "cellfm_multi_INT21_holdout",
    "single": ROOT / "checkpoints" / "kidney_hest" / "st_superres" / "cellfm_single_INT18_to_INT21",
}

BATCH_SIZE = 32
NUM_WORKERS = 0
GENE_CHUNK_SIZE = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Slides:", SLIDE_IDS)
print("Models:", list(MODEL_SPECS))


Device: cuda
Slides: ['GSM6415706_GLMF1', 'GSM6415706_GLMF2']
Models: ['multi', 'single']


C:\ProgramData\anaconda3\envs\scellst_bench\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def display_df(df):
    if display is not None:
        display(df)
    else:
        print(df)


def req(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def pearson1d(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    m = np.isfinite(a) & np.isfinite(b)
    a = a[m]
    b = b[m]
    if a.size < 3:
        return np.nan
    a = a - a.mean()
    b = b - b.mean()
    den = float(np.sqrt((a * a).sum()) * np.sqrt((b * b).sum()))
    return np.nan if den == 0.0 else float((a * b).sum() / den)


def pearson_by_gene(pred, true):
    pred = np.asarray(pred, dtype=np.float32)
    true = np.asarray(true, dtype=np.float32)
    n = pred.shape[0]
    sp = pred.sum(0)
    st = true.sum(0)
    spt = (pred * true).sum(0)
    sp2 = (pred * pred).sum(0)
    st2 = (true * true).sum(0)
    num = spt - (sp * st / n)
    den = np.sqrt(sp2 - (sp ** 2) / n) * np.sqrt(st2 - (st ** 2) / n)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = num / den
    out[~np.isfinite(out)] = np.nan
    return out


def pearson_by_spot(pred, true):
    pred = np.asarray(pred, dtype=np.float32)
    true = np.asarray(true, dtype=np.float32)
    n = pred.shape[1]
    sp = pred.sum(1)
    st = true.sum(1)
    spt = (pred * true).sum(1)
    sp2 = (pred * pred).sum(1)
    st2 = (true * true).sum(1)
    num = spt - (sp * st / n)
    den = np.sqrt(sp2 - (sp ** 2) / n) * np.sqrt(st2 - (st ** 2) / n)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = num / den
    out[~np.isfinite(out)] = np.nan
    return out


def load_cache_slide(slide_id: str):
    d = CACHE_ROOT / slide_id
    counts = np.load(req(d / "counts.npy"), mmap_mode="r")
    locs = np.load(req(d / "locs.npy"), mmap_mode="r")
    genes = [x.strip() for x in req(d / "genes.txt").read_text(encoding="utf-8").splitlines() if x.strip()]
    spot_ids = [x.strip() for x in req(d / "spot_ids.txt").read_text(encoding="utf-8").splitlines() if x.strip()]
    radius = float(req(d / "radius.txt").read_text(encoding="utf-8").strip())
    with open(req(d / "embeddings-hist.pickle"), "rb") as f:
        embs = pickle.load(f)
    embs = np.concatenate([embs["cls"], embs["sub"], embs["rgb"]]).transpose(1, 2, 0)
    embs = np.asarray(embs, dtype=np.float32)
    return embs, counts, locs, genes, spot_ids, radius


def get_disk_mask(radius):
    radius_ceil = int(np.ceil(radius))
    locs = np.meshgrid(
        np.arange(-radius_ceil, radius_ceil + 1),
        np.arange(-radius_ceil, radius_ceil + 1),
        indexing="ij",
    )
    locs = np.stack(locs, axis=-1)
    distsq = (locs ** 2).sum(-1)
    return distsq <= radius ** 2


class SuperresSpotDataset(Dataset):
    def __init__(self, embs, cnts, locs_px, radius_px, gene_ids, spot_ids, grid_factor=16):
        super().__init__()
        self.embs = embs
        self.cnts = cnts
        self.gene_ids = gene_ids
        locs_px = np.asarray(locs_px, dtype=np.float32)
        self.locs_px = locs_px.copy()
        locs_grid = np.round(locs_px / float(grid_factor)).astype(np.int64)
        self.radius = float(radius_px) / float(grid_factor)
        self.mask = get_disk_mask(self.radius)
        self.mask_size = int(self.mask.sum())

        h, w = embs.shape[:2]
        r0, r1 = self._patch_offsets()
        valid = (
            (locs_grid[:, 0] + r0[0] >= 0)
            & (locs_grid[:, 0] + r0[1] <= h)
            & (locs_grid[:, 1] + r1[0] >= 0)
            & (locs_grid[:, 1] + r1[1] <= w)
        )
        finite_center = np.isfinite(embs[locs_grid[:, 0], locs_grid[:, 1]]).all(-1)
        valid &= finite_center

        self.keep_idx = np.where(valid)[0]
        self.locs = locs_grid[valid]
        self.locs_px = self.locs_px[valid]
        self.cnts = cnts[valid]
        self.spot_ids = [spot_ids[i] for i in self.keep_idx]

        sums = self.cnts.sum(axis=1)
        self.size_factors = (sums / (sums.mean() + 1e-8)).astype(np.float32)

    def _patch_offsets(self):
        shape = np.array(self.mask.shape)
        center = shape // 2
        r = np.stack([-center, shape - center], -1)
        return r[0], r[1]

    def __len__(self):
        return len(self.locs)

    def __getitem__(self, idx):
        i, j = self.locs[idx]
        r0, r1 = self._patch_offsets()
        patch = self.embs[
            i + r0[0] : i + r0[1],
            j + r1[0] : j + r1[1],
        ]
        x = patch if self.mask.all() else patch[self.mask]
        x = torch.from_numpy(x.astype(np.float32))
        y = torch.from_numpy(self.cnts[idx].astype(np.float32))
        size_factor = float(self.size_factors[idx])
        return {
            KEYS.X: x,
            KEYS.Y_BAG: y,
            KEYS.SIZE_FACTOR: size_factor,
            KEYS.GENE_IDS: self.gene_ids,
        }


In [3]:
vocab = load_gene_vocab(GENE_VOCAB)


def build_model(input_dim: int, n_genes: int):
    return MorphoCellfmMIL(
        input_dim=input_dim,
        n_genes=n_genes,
        cellfm_dim=1536,
        cellfm_layers=2,
        cellfm_heads=48,
        cellfm_checkpoint=str(CELLFM_CKPT),
        freeze_cellfm=True,
        use_mock=False,
        use_retention=True,
        vocab_size=len(vocab),
        dropout=0.1,
        aggregation="mean",
        dispersion="gene",
    ).to(device)


def run_prediction(slide_id: str, model_key: str, force_rerun: bool = False, show_progress: bool = True):
    model_dir = MODEL_SPECS[model_key]
    result_dir = RESULT_ROOT / slide_id / model_key
    result_dir.mkdir(parents=True, exist_ok=True)

    pred_path = result_dir / "pred_bag.npy"
    true_path = result_dir / "true_bag.npy"
    corr_path = result_dir / "gene_corr_all.npy"
    spot_path = result_dir / "spot_ids.npy"
    genes_path = result_dir / "kept_genes.txt"
    coords_path = result_dir / "locs_px.npy"
    metrics_path = result_dir / "metrics.json"

    common_genes = json.loads((model_dir / "common_genes.json").read_text(encoding="utf-8"))
    embs, counts_all, locs, genes_all, spot_ids, radius = load_cache_slide(slide_id)

    gene_to_idx = {g: i for i, g in enumerate(genes_all)}
    missing = [g for g in common_genes if g not in gene_to_idx]
    if missing:
        raise ValueError(f"{slide_id}/{model_key}: missing genes {missing[:10]}")
    idx = np.array([gene_to_idx[g] for g in common_genes], dtype=np.int64)
    counts = np.asarray(counts_all[:, idx], dtype=np.float32)

    kept_genes, gene_ids = map_genes_to_vocab(common_genes, vocab)
    if len(kept_genes) != len(common_genes):
        raise ValueError(f"{slide_id}/{model_key}: vocab mapping mismatch {len(kept_genes)} vs {len(common_genes)}")

    embs_norm, _, _ = normalize_slide(embs, counts)
    ds = SuperresSpotDataset(embs_norm, counts, locs, radius, gene_ids, spot_ids)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=mil_collate)

    cache_ready = all([
        pred_path.exists(),
        true_path.exists(),
        corr_path.exists(),
        spot_path.exists(),
        genes_path.exists(),
        coords_path.exists(),
    ])

    if cache_ready and not force_rerun:
        print(f"Using cached prediction: {slide_id} | {model_key}")
        pred_bag = np.load(pred_path)
        true_bag = np.load(true_path)
        gene_corr = np.load(corr_path)
    else:
        sample_batch = next(iter(loader))
        input_dim = int(sample_batch[KEYS.X].shape[-1])
        model = build_model(input_dim=input_dim, n_genes=len(kept_genes))
        model.load_state_dict(torch.load(model_dir / "best_model.pt", map_location=device))
        model.eval()

        pred_bag = np.zeros((len(ds), len(kept_genes)), dtype=np.float32)
        true_bag = np.zeros((len(ds), len(kept_genes)), dtype=np.float32)
        gene_chunks = [
            list(range(i, min(i + GENE_CHUNK_SIZE, len(kept_genes))))
            for i in range(0, len(kept_genes), GENE_CHUNK_SIZE)
        ]

        offset = 0
        batch_iter = loader
        if show_progress:
            batch_iter = tqdm(loader, total=len(loader), desc=f"{slide_id} | {model_key}", leave=False)
        with torch.no_grad():
            for batch in batch_iter:
                for k in batch:
                    batch[k] = batch[k].to(device)
                bs = batch[KEYS.Y_BAG].shape[0]
                true_bag[offset:offset + bs] = batch[KEYS.Y_BAG].cpu().numpy()
                chunk_preds = []
                for chunk in gene_chunks:
                    gidx = torch.as_tensor(chunk, device=device)
                    batch_chunk = {
                        KEYS.X: batch[KEYS.X],
                        KEYS.PTR_BAG_INSTANCE: batch[KEYS.PTR_BAG_INSTANCE],
                        KEYS.SIZE_FACTOR: batch[KEYS.SIZE_FACTOR],
                        KEYS.GENE_IDS: batch[KEYS.GENE_IDS][gidx],
                    }
                    mu_bag, _ = model(batch_chunk)
                    chunk_preds.append(mu_bag.cpu().numpy())
                pred_bag[offset:offset + bs] = np.concatenate(chunk_preds, axis=1)
                offset += bs
                if show_progress and hasattr(batch_iter, "set_postfix"):
                    batch_iter.set_postfix({"spots": f"{offset}/{len(ds)}", "chunks": len(gene_chunks)})

        gene_corr = pearson_by_gene(pred_bag, true_bag)
        np.save(pred_path, pred_bag)
        np.save(true_path, true_bag)
        np.save(corr_path, gene_corr)
        np.save(spot_path, np.asarray(ds.spot_ids, dtype=object))
        np.save(coords_path, np.asarray(ds.locs_px, dtype=np.float32))
        genes_path.write_text("\n".join(kept_genes), encoding="utf-8")

    metrics = {
        "slide_id": slide_id,
        "model_key": model_key,
        "n_spots": int(pred_bag.shape[0]),
        "n_genes": int(pred_bag.shape[1]),
        "mae": float(np.mean(np.abs(pred_bag - true_bag))),
        "rmse": float(np.sqrt(np.mean((pred_bag - true_bag) ** 2))),
        "mean_pearson": float(np.nanmean(gene_corr)),
        "median_pearson": float(np.nanmedian(gene_corr)),
        "radius_px": float(radius),
        "mask_size": int(get_disk_mask(radius / 16.0).sum()),
        "result_dir": str(result_dir),
    }
    metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Finished: {slide_id} | {model_key} | spots={metrics['n_spots']} | genes={metrics['n_genes']}")
    return metrics


def run_batch(slide_ids, model_keys, force_rerun: bool = False, show_progress: bool = True):
    rows = []
    cases = [(slide_id, model_key) for slide_id in slide_ids for model_key in model_keys]
    case_iter = cases
    if show_progress:
        case_iter = tqdm(cases, total=len(cases), desc="Cases", leave=True)
    for slide_id, model_key in case_iter:
        print(f"Running {slide_id} | {model_key}")
        rows.append(run_prediction(slide_id, model_key, force_rerun=force_rerun, show_progress=show_progress))
    return pd.DataFrame(rows).sort_values(["slide_id", "model_key"]).reset_index(drop=True)


## Config

Edit only the next cell. Then run the prediction cell and the final export cell.


In [4]:
TARGET_MODEL_KEY = "multi"
TARGET_SLIDE_IDS = list(SLIDE_IDS)
FORCE_RERUN = False
SHOW_PROGRESS = True

TARGET_MARKERS = None  # Set a manual list like ["PAX8", "CA9"] to override automatic gene selection
TOP_N_PER_SLIDE = 4
SPATIAL_SCALE_MODE = "separate"  # separate or shared
SPATIAL_SPOT_SIZE = 8
PAIRS_PER_ROW = 2

SAVE_FIGURES = True
FIG_FORMAT = "pdf"
DISPLAY_FIGURES = False

print("Slides:", TARGET_SLIDE_IDS)
print("Model:", TARGET_MODEL_KEY)
print("Force rerun:", FORCE_RERUN)
print("Show progress:", SHOW_PROGRESS)
print("Manual display genes:", TARGET_MARKERS)
print("Top genes per slide:", TOP_N_PER_SLIDE)
print("Spatial scale mode:", SPATIAL_SCALE_MODE)
print("Spatial spot size:", SPATIAL_SPOT_SIZE)
print("Pairs per row:", PAIRS_PER_ROW)
print("Save figures:", SAVE_FIGURES)
print("Figure format:", FIG_FORMAT)
print("Display figures inline:", DISPLAY_FIGURES)


Slides: ['GSM6415706_GLMF1', 'GSM6415706_GLMF2']
Model: multi
Force rerun: False
Show progress: True
Manual display genes: None
Top genes per slide: 4
Spatial scale mode: separate
Spatial spot size: 8
Pairs per row: 2
Save figures: True
Figure format: pdf
Display figures inline: False


In [5]:
prediction_summary_df = run_batch(
    TARGET_SLIDE_IDS,
    [TARGET_MODEL_KEY],
    force_rerun=FORCE_RERUN,
    show_progress=SHOW_PROGRESS,
)
display_df(prediction_summary_df)


Cases:   0%|          | 0/2 [00:00<?, ?it/s]

Running GSM6415706_GLMF1 | multi
Using cached prediction: GSM6415706_GLMF1 | multi


Cases:  50%|█████     | 1/2 [00:06<00:06,  6.08s/it]

Finished: GSM6415706_GLMF1 | multi | spots=3199 | genes=17512
Running GSM6415706_GLMF2 | multi
Using cached prediction: GSM6415706_GLMF2 | multi


Cases: 100%|██████████| 2/2 [00:12<00:00,  6.29s/it]

Finished: GSM6415706_GLMF2 | multi | spots=3880 | genes=17512


,slide_id,model_key,n_spots,n_genes,mae,rmse,mean_pearson,median_pearson,radius_px,mask_size,result_dir
0,GSM6415706_GLMF1,multi,3199,17512,0.910066,10.475981,0.293586,0.297760,55.836288,37,E:\Morpho-FM\results\other_kidney\GSM6415706_G...
1,GSM6415706_GLMF2,multi,3880,17512,0.624741,4.351251,0.138244,0.122453,55.878102,37,E:\Morpho-FM\results\other_kidney\GSM6415706_G...


## Export Tables and Save Spatial PDF Figures

The next cells keep the original gene-selection logic unchanged:
- each slide selects its own top 4 genes
- the final displayed genes are the union of those per-slide lists
- each slide is saved as one PDF figure on the tissue background


In [6]:
RCC_MARKERS = ["PAX8", "CA9", "MME", "NDUFA4L2", "KRT7", "AMACR", "KIT", "FOXI1", "RHCG", "TFE3", "TFEB", "GPNMB", "FH", "SDHB", "ALK", "SMARCB1"]

BIOLOGICAL_GENE_INFO = {
    "PAX8": ("RCC_origin", "renal epithelial lineage / kidney origin", 5),
    "CA9": ("ccRCC", "hypoxia-driven clear cell RCC marker", 5),
    "MME": ("ccRCC", "CD10 proximal-tubule / clear cell RCC associated marker", 4),
    "NDUFA4L2": ("ccRCC", "hypoxia program in clear cell RCC", 5),
    "KRT7": ("pRCC_or_chRCC", "papillary or chromophobe RCC associated keratin program", 5),
    "AMACR": ("pRCC", "papillary RCC marker", 5),
    "KIT": ("chRCC", "chromophobe RCC / intercalated-cell marker", 5),
    "FOXI1": ("chRCC", "chromophobe RCC transcriptional program", 5),
    "RHCG": ("chRCC", "chromophobe RCC / intercalated-cell transport program", 5),
    "TFE3": ("TFE3_TFEB_RCC", "MiT family translocation RCC driver", 5),
    "TFEB": ("TFE3_TFEB_RCC", "MiT family translocation RCC driver", 5),
    "GPNMB": ("TFE3_TFEB_RCC", "MiT family translocation RCC downstream program", 4),
    "FH": ("special_defect", "FH-deficient RCC biology", 5),
    "SDHB": ("special_defect", "SDH-deficient RCC biology", 5),
    "ALK": ("special_defect", "ALK-rearranged RCC biology", 4),
    "SMARCB1": ("special_defect", "SMARCB1-deficient renal tumor biology", 4),
    "AQP1": ("proximal_tubule", "proximal-tubule / clear-cell lineage signal", 3),
    "SLC34A1": ("proximal_tubule", "proximal-tubule differentiation signal", 3),
    "EPCAM": ("epithelial", "epithelial tumor component", 2),
    "VIM": ("mesenchymal", "mesenchymal / invasive tumor program", 2),
    "HAVCR1": ("injury", "tubular injury / aggressive renal program", 2),
    "LCN2": ("injury", "injury-associated renal epithelial signal", 2),
    "VEGFA": ("angiogenesis", "angiogenic / hypoxic renal tumor biology", 3),
    "VCAM1": ("stress_inflammation", "stress / inflammatory renal microenvironment", 2),
}


def load_slide_background(slide_id: str):
    img_path = req(CACHE_ROOT / slide_id / "he.jpg")
    img = plt.imread(img_path)
    return img


def load_saved_result(slide_id: str, model_key: str):
    result_dir = RESULT_ROOT / slide_id / model_key
    pred = np.load(req(result_dir / "pred_bag.npy"))
    true = np.load(req(result_dir / "true_bag.npy"))
    gene_corr = np.load(req(result_dir / "gene_corr_all.npy"))
    spot_ids = np.load(req(result_dir / "spot_ids.npy"), allow_pickle=True).tolist()
    locs = np.load(req(result_dir / "locs_px.npy"))
    genes = [x.strip() for x in req(result_dir / "kept_genes.txt").read_text(encoding="utf-8").splitlines() if x.strip()]
    metrics = {}
    metrics_path = result_dir / "metrics.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    return result_dir, pred, true, gene_corr, spot_ids, locs, genes, metrics


def export_comparison_tables(slide_id: str, model_key: str, marker_genes=None):
    marker_genes = marker_genes or RCC_MARKERS
    result_dir, pred, true, gene_corr, spot_ids, locs, genes, metrics = load_saved_result(slide_id, model_key)

    abs_err = np.abs(pred - true)
    sq_err = (pred - true) ** 2
    true_nonzero_frac = (np.asarray(true) > 0).mean(axis=0)
    pred_nonzero_frac = (np.asarray(pred) > 0).mean(axis=0)

    bio_group = []
    bio_note = []
    bio_priority = []
    for gene in genes:
        group, note, priority = BIOLOGICAL_GENE_INFO.get(gene, ("other", "", 0))
        bio_group.append(group)
        bio_note.append(note)
        bio_priority.append(priority)

    gene_df = pd.DataFrame({
        "gene": genes,
        "panel": [BIOLOGICAL_GENE_INFO.get(g, ("other", "", 0))[0] for g in genes],
        "pearson": gene_corr,
        "mae": abs_err.mean(axis=0),
        "rmse": np.sqrt(sq_err.mean(axis=0)),
        "pred_mean": pred.mean(axis=0),
        "true_mean": true.mean(axis=0),
        "pred_nonzero_frac": pred_nonzero_frac,
        "true_nonzero_frac": true_nonzero_frac,
        "bio_group": bio_group,
        "bio_note": bio_note,
        "bio_priority": bio_priority,
        "is_rcc_marker": [g in RCC_MARKERS for g in genes],
    })
    gene_df.to_csv(result_dir / "gene_metrics.csv", index=False)

    spot_corr = pearson_by_spot(pred, true)
    spot_df = pd.DataFrame({
        "spot_id": spot_ids,
        "y": locs[:, 0],
        "x": locs[:, 1],
        "pearson": spot_corr,
        "mae": abs_err.mean(axis=1),
        "rmse": np.sqrt(sq_err.mean(axis=1)),
    })
    spot_df.to_csv(result_dir / "spot_metrics.csv", index=False)

    marker_df = gene_df[gene_df["gene"].isin(marker_genes)].copy()
    marker_df = marker_df.sort_values(["panel", "pearson", "true_mean"], ascending=[True, False, False]).reset_index(drop=True)
    marker_df.to_csv(result_dir / "marker_metrics.csv", index=False)

    metrics.update({
        "global_pearson": float(pearson1d(pred.reshape(-1), true.reshape(-1))),
        "spot_mean_pearson": float(np.nanmean(spot_corr)),
        "spot_median_pearson": float(np.nanmedian(spot_corr)),
        "gene_top20_mean_pearson": float(gene_df["pearson"].sort_values(ascending=False).head(20).mean()),
    })
    (result_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    return metrics


def recommend_display_genes(slide_id: str, model_key: str, top_n: int = 6, manual_genes=None, prefer_biological: bool = True):
    export_comparison_tables(slide_id, model_key)
    result_dir = RESULT_ROOT / slide_id / model_key
    gene_df = pd.read_csv(result_dir / "gene_metrics.csv")
    gene_df = gene_df[np.isfinite(gene_df["pearson"])].copy()

    if manual_genes is not None:
        out = gene_df[gene_df["gene"].isin(manual_genes)].copy()
        out["selection_reason"] = "manual override"
        out = out.sort_values(["bio_priority", "pearson", "true_mean"], ascending=[False, False, False]).reset_index(drop=True)
        out.to_csv(result_dir / "recommended_display_genes.csv", index=False)
        return out

    if len(gene_df) == 0:
        return gene_df

    mean_cut = max(0.01, float(gene_df["true_mean"].quantile(0.40)))
    nz_cut = max(0.02, float(gene_df["true_nonzero_frac"].quantile(0.40)))
    filtered = gene_df[(gene_df["true_mean"] >= mean_cut) & (gene_df["true_nonzero_frac"] >= nz_cut)].copy()
    if len(filtered) < top_n:
        filtered = gene_df[gene_df["true_nonzero_frac"] >= min(0.01, nz_cut)].copy()
    if len(filtered) < top_n:
        filtered = gene_df.copy()

    filtered = filtered.sort_values(["pearson", "true_nonzero_frac", "true_mean"], ascending=[False, False, False]).reset_index(drop=True)

    parts = []
    if prefer_biological:
        biological = filtered[filtered["bio_priority"] > 0].copy()
        biological["selection_reason"] = "high-correlation + biologically interpretable"
        parts.append(biological)
    fallback = filtered.copy()
    fallback["selection_reason"] = "high-correlation fallback"
    parts.append(fallback)

    out = pd.concat(parts, ignore_index=True)
    out = out.drop_duplicates(subset=["gene"]).reset_index(drop=True).head(top_n)
    out.to_csv(result_dir / "recommended_display_genes.csv", index=False)
    return out


def resolve_display_genes(slide_id: str, model_key: str, genes_to_plot=None, top_n: int = 4):
    if genes_to_plot is not None:
        return list(genes_to_plot)
    rec_df = recommend_display_genes(slide_id, model_key, top_n=top_n)
    return rec_df["gene"].tolist()


def resolve_union_display_genes(slide_ids, model_key: str, genes_to_plot=None, top_n_per_slide: int = 4, verbose: bool = True):
    if genes_to_plot is not None:
        union_genes = list(dict.fromkeys(list(genes_to_plot)))
        per_slide = {slide_id: union_genes for slide_id in slide_ids}
        if verbose:
            print(f"Manual union display genes: {union_genes}")
        return union_genes, per_slide

    per_slide = {}
    union_genes = []
    for slide_id in slide_ids:
        slide_genes = resolve_display_genes(slide_id, model_key, top_n=top_n_per_slide)
        per_slide[slide_id] = slide_genes
        for gene in slide_genes:
            if gene not in union_genes:
                union_genes.append(gene)

    if verbose:
        print(f"Per-slide display genes | model={model_key}")
        for slide_id, slide_genes in per_slide.items():
            print(f"  {slide_id}: {slide_genes}")
        print(f"Union display genes ({len(union_genes)}): {union_genes}")
    return union_genes, per_slide

def load_metric_tables(slide_id: str, model_key: str, top_n_recommended: int = 8):
    export_comparison_tables(slide_id, model_key)
    result_dir = RESULT_ROOT / slide_id / model_key
    metrics = json.loads((result_dir / "metrics.json").read_text(encoding="utf-8"))
    gene_df = pd.read_csv(result_dir / "gene_metrics.csv")
    marker_df = pd.read_csv(result_dir / "marker_metrics.csv")
    recommended_df = recommend_display_genes(slide_id, model_key, top_n=top_n_recommended)
    return result_dir, metrics, gene_df, marker_df, recommended_df


In [7]:
FIG_SUBDIR = "figures"
FIG_DPI = 300


def get_figure_dir(slide_id: str, model_key: str) -> Path:
    d = RESULT_ROOT / slide_id / model_key / FIG_SUBDIR
    d.mkdir(parents=True, exist_ok=True)
    return d


def safe_name(text: str) -> str:
    return str(text).replace("/", "_").replace("\\", "_").replace(" ", "_")


def get_figure_format() -> str:
    fmt = str(globals().get("FIG_FORMAT", "pdf")).strip().lower().lstrip(".")
    return fmt or "pdf"


def compute_color_limits(values, q_low: float = 0.01, q_high: float = 0.99):
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    if not finite.any():
        return 0.0, 1.0
    vals = values[finite]
    lo, hi = np.quantile(vals, [q_low, q_high])
    lo = float(lo)
    hi = float(hi)
    if hi <= lo:
        hi = lo + 1e-6
    return lo, hi


def plot_marker_maps(
    slide_id: str,
    model_key: str,
    genes_to_plot=None,
    top_n: int = 4,
    background_alpha: float = 1.0,
    scale_mode: str = "separate",
    spot_size: float = 12,
    save: bool = False,
    out_dir=None,
    dpi: int = FIG_DPI,
    pairs_per_row: int = 2,
    show: bool = False,
):
    genes_to_plot = resolve_display_genes(slide_id, model_key, genes_to_plot=genes_to_plot, top_n=top_n)

    result_dir = RESULT_ROOT / slide_id / model_key
    pred = np.load(result_dir / "pred_bag.npy")
    true = np.load(result_dir / "true_bag.npy")
    locs = np.load(result_dir / "locs_px.npy")
    kept_genes = [x.strip() for x in (result_dir / "kept_genes.txt").read_text(encoding="utf-8").splitlines() if x.strip()]
    bg = load_slide_background(slide_id)
    h, w = bg.shape[:2]

    plot_items = []
    for gene in genes_to_plot:
        if gene not in kept_genes:
            print(f"[SKIP] {slide_id}/{model_key}/{gene}")
            continue
        gi = kept_genes.index(gene)
        true_vals = np.asarray(true[:, gi], dtype=np.float32)
        pred_vals = np.asarray(pred[:, gi], dtype=np.float32)
        r = pearson1d(true_vals, pred_vals)

        if scale_mode == "shared":
            shared_lo, shared_hi = compute_color_limits(np.concatenate([true_vals, pred_vals]))
            panel_limits = [(shared_lo, shared_hi), (shared_lo, shared_hi)]
        else:
            panel_limits = [compute_color_limits(true_vals), compute_color_limits(pred_vals)]

        plot_items.append((gene, true_vals, pred_vals, r, panel_limits))

    if not plot_items:
        print(f"[SKIP] no valid genes to plot for {slide_id}/{model_key}")
        return {"genes_plotted": [], "saved_path": None}

    pairs_per_row = max(1, min(int(pairs_per_row), len(plot_items)))
    nrows = int(np.ceil(len(plot_items) / pairs_per_row))
    ncols = pairs_per_row * 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.1 * nrows), dpi=220)
    axes = np.array(axes, dtype=object).reshape(nrows, ncols)
    fig.suptitle(f"{slide_id} | {model_key} | {len(plot_items)} genes | scale={scale_mode}", y=0.995)

    for idx, (gene, true_vals, pred_vals, r, panel_limits) in enumerate(plot_items):
        row = idx // pairs_per_row
        col0 = (idx % pairs_per_row) * 2
        gene_axes = [axes[row, col0], axes[row, col0 + 1]]
        for ax, vals, title, limits in zip(
            gene_axes,
            [true_vals, pred_vals],
            ["True", f"Pred ({model_key})"],
            panel_limits,
        ):
            lo, hi = limits
            ax.imshow(bg, alpha=background_alpha)
            sc = ax.scatter(
                locs[:, 1],
                locs[:, 0],
                c=vals,
                s=spot_size,
                cmap="viridis",
                vmin=lo,
                vmax=hi,
                linewidths=0,
                alpha=0.90,
            )
            ax.set_title("{} | {}\nr={:.3f} | q01-q99: [{:.3g}, {:.3g}]".format(gene, title, r, lo, hi), fontsize=10)
            ax.set_xlim(0, w)
            ax.set_ylim(h, 0)
            ax.axis("off")
            fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.01)

    total_pairs = nrows * pairs_per_row
    for idx in range(len(plot_items), total_pairs):
        row = idx // pairs_per_row
        col0 = (idx % pairs_per_row) * 2
        axes[row, col0].set_axis_off()
        axes[row, col0 + 1].set_axis_off()

    plt.tight_layout(rect=[0, 0, 1, 0.985])

    saved_path = None
    if save:
        if out_dir is None:
            out_dir = get_figure_dir(slide_id, model_key)
        fig_format = get_figure_format()
        saved_path = out_dir / f"marker_maps_top{len(plot_items)}_{scale_mode}.{fig_format}"
        fig.savefig(saved_path, dpi=dpi, bbox_inches="tight", format=fig_format)
        print(f"[SAVED] {saved_path}")

    if show:
        plt.show()
    plt.close(fig)
    return {"genes_plotted": [item[0] for item in plot_items], "saved_path": str(saved_path) if saved_path is not None else None}


def show_case_visuals(
    slide_id: str,
    model_key: str,
    genes_to_plot=None,
    top_n: int = 4,
    scale_mode: str = "separate",
    spot_size: float = 12,
    save: bool = False,
    pairs_per_row: int = 2,
    show: bool = False,
):
    display_genes = resolve_display_genes(slide_id, model_key, genes_to_plot=genes_to_plot, top_n=top_n)
    print(f"Display genes for {slide_id} | {model_key}: {display_genes}")
    plot_info = plot_marker_maps(
        slide_id,
        model_key,
        genes_to_plot=display_genes,
        top_n=top_n,
        scale_mode=scale_mode,
        spot_size=spot_size,
        save=save,
        pairs_per_row=pairs_per_row,
        show=show,
    )
    return {
        "display_genes": display_genes,
        "genes_plotted": plot_info["genes_plotted"],
        "saved_path": plot_info["saved_path"],
    }


def show_all_slides_for_model(
    model_key: str,
    slide_ids=None,
    genes_to_plot=None,
    top_n: int = 4,
    scale_mode: str = "separate",
    spot_size: float = 12,
    save: bool = False,
    pairs_per_row: int = 2,
    show: bool = False,
    per_slide_display_genes=None,
):
    slide_ids = list(slide_ids or SLIDE_IDS)
    rows = []
    for slide_id in slide_ids:
        print("=" * 80)
        print(f"Slide: {slide_id} | Model: {model_key}")
        _, metrics, _, _, _ = load_metric_tables(slide_id, model_key, top_n_recommended=max(8, top_n))
        plot_result = show_case_visuals(
            slide_id,
            model_key,
            genes_to_plot=genes_to_plot,
            top_n=top_n,
            scale_mode=scale_mode,
            spot_size=spot_size,
            save=save,
            pairs_per_row=pairs_per_row,
            show=show,
        )
        rows.append({
            "slide_id": slide_id,
            "model_key": model_key,
            "per_slide_display_genes": ", ".join((per_slide_display_genes or {}).get(slide_id, [])),
            "union_display_genes": ", ".join(plot_result["display_genes"]),
            "genes_plotted": ", ".join(plot_result["genes_plotted"]),
            "mean_pearson": metrics.get("mean_pearson"),
            "global_pearson": metrics.get("global_pearson"),
            "spot_mean_pearson": metrics.get("spot_mean_pearson"),
            "top20_gene_mean_pearson": metrics.get("gene_top20_mean_pearson"),
            "saved_figure": plot_result["saved_path"],
        })
    summary_df = pd.DataFrame(rows)
    print("=" * 80)
    return summary_df


In [8]:
union_display_genes, per_slide_display_genes = resolve_union_display_genes(
    TARGET_SLIDE_IDS,
    TARGET_MODEL_KEY,
    genes_to_plot=TARGET_MARKERS,
    top_n_per_slide=TOP_N_PER_SLIDE,
)

selection_export_path = RESULT_ROOT / f"other_kidney_{TARGET_MODEL_KEY}_union_display_genes.json"
selection_export_path.write_text(
    json.dumps(
        {
            "model_key": TARGET_MODEL_KEY,
            "slide_ids": TARGET_SLIDE_IDS,
            "per_slide_display_genes": per_slide_display_genes,
            "union_display_genes": union_display_genes,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
print(f"[SAVED] {selection_export_path}")

all_slides_selected_model_df = show_all_slides_for_model(
    TARGET_MODEL_KEY,
    slide_ids=TARGET_SLIDE_IDS,
    genes_to_plot=union_display_genes,
    top_n=len(union_display_genes),
    scale_mode=SPATIAL_SCALE_MODE,
    spot_size=SPATIAL_SPOT_SIZE,
    save=SAVE_FIGURES,
    pairs_per_row=PAIRS_PER_ROW,
    show=DISPLAY_FIGURES,
    per_slide_display_genes=per_slide_display_genes,
)

summary_export_path = RESULT_ROOT / f"other_kidney_{TARGET_MODEL_KEY}_figure_summary.csv"
all_slides_selected_model_df.to_csv(summary_export_path, index=False)
print(f"[SAVED] {summary_export_path}")
display_df(all_slides_selected_model_df)


Per-slide display genes | model=multi
  GSM6415706_GLMF1: ['SMARCB1', 'SDHB', 'TFE3', 'TFEB']
  GSM6415706_GLMF2: ['AQP1', 'VCAM1', 'SMARCB1', 'SDHB']
Union display genes (6): ['SMARCB1', 'SDHB', 'TFE3', 'TFEB', 'AQP1', 'VCAM1']
[SAVED] E:\Morpho-FM\results\other_kidney\other_kidney_multi_union_display_genes.json
Slide: GSM6415706_GLMF1 | Model: multi
Display genes for GSM6415706_GLMF1 | multi: ['SMARCB1', 'SDHB', 'TFE3', 'TFEB', 'AQP1', 'VCAM1']
[SAVED] E:\Morpho-FM\results\other_kidney\GSM6415706_GLMF1\multi\figures\marker_maps_top6_separate.pdf
Slide: GSM6415706_GLMF2 | Model: multi
Display genes for GSM6415706_GLMF2 | multi: ['SMARCB1', 'SDHB', 'TFE3', 'TFEB', 'AQP1', 'VCAM1']
[SAVED] E:\Morpho-FM\results\other_kidney\GSM6415706_GLMF2\multi\figures\marker_maps_top6_separate.pdf
[SAVED] E:\Morpho-FM\results\other_kidney\other_kidney_multi_figure_summary.csv


,slide_id,model_key,per_slide_display_genes,union_display_genes,genes_plotted,mean_pearson,global_pearson,spot_mean_pearson,top20_gene_mean_pearson,saved_figure
0,GSM6415706_GLMF1,multi,"SMARCB1, SDHB, TFE3, TFEB","SMARCB1, SDHB, TFE3, TFEB, AQP1, VCAM1","SMARCB1, SDHB, TFE3, TFEB, AQP1, VCAM1",0.293586,0.503822,0.657781,0.860440,E:\Morpho-FM\results\other_kidney\GSM6415706_G...
1,GSM6415706_GLMF2,multi,"AQP1, VCAM1, SMARCB1, SDHB","SMARCB1, SDHB, TFE3, TFEB, AQP1, VCAM1","SMARCB1, SDHB, TFE3, TFEB, AQP1, VCAM1",0.138244,0.467365,0.500480,0.700781,E:\Morpho-FM\results\other_kidney\GSM6415706_G...
